# 40 Submission 12

Final submission using the Experiment 39 Seed 7 CatBoost configuration.

## 1. Load Data

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

TRAIN_PATH = '../data/train.csv'
TEST_PATH = '../data/test.csv'
TARGET = 'Will_Buy_EV'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y = train[TARGET].astype(str).str.strip().map({'No': 0, 'Yes': 1}).astype(int)

test_ids = test['id'].copy()

X_train = train.drop(columns=[TARGET, 'id']).copy()
X_test = test.drop(columns=['id']).copy()

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

Train shape: (668665, 13)
Test shape: (286571, 13)


## 2. Feature Engineering

In [2]:
categorical_cols = [
    'Gender', 'City_Type', 'Current_Car_Type',
    'Home_Charging_Possible', 'Subsidy_Available',
    'Range_Anxiety_Level'
]

def engineer_features(X):
    X = X.copy()

    sub = X['Subsidy_Available'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)
    home = X['Home_Charging_Possible'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)

    X['Subsidy_x_EnvConcern'] = sub * X['Environmental_Concern_Level']
    X['Subsidy_x_Income'] = sub * X['Annual_Income_USD']
    X['Subsidy_x_HomeCharging'] = sub * home

    value_identity_cols = [
        'Age',
        'Annual_Income_USD',
        'Daily_Commute_km',
        'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work'
    ]

    for col in value_identity_cols:
        X[f'{col}__value_id'] = X[col].astype('string').fillna('__MISSING__')

    return X

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)

value_identity_cols = [
    'Age__value_id',
    'Annual_Income_USD__value_id',
    'Daily_Commute_km__value_id',
    'Charging_Stations_Near_Home__value_id',
    'Charging_Stations_Near_Work__value_id'
]

cat_identity_cols = categorical_cols + value_identity_cols

for col in cat_identity_cols:
    X_train[col] = X_train[col].astype('string').fillna('__MISSING__')
    X_test[col] = X_test[col].astype('string').fillna('__MISSING__')

X_test = X_test[X_train.columns]

print('Model features:', X_train.shape[1])
print('Categorical features:', len(cat_identity_cols))

Model features: 21
Categorical features: 11


## 3. Train Seed 7 CatBoost

In [3]:
model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=3,
    random_strength=1,
    bootstrap_type='Bayesian',
    bagging_temperature=1,
    random_seed=7,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

model.fit(
    X_train,
    y,
    cat_features=cat_identity_cols,
    verbose=False
)

print('Seed 7 CatBoost trained on full training data.')

Seed 7 CatBoost trained on full training data.


## 4. Predict Test Set

In [4]:
test_pred = model.predict_proba(X_test)[:, 1]

print('Prediction min:', test_pred.min())
print('Prediction max:', test_pred.max())
print('Prediction mean:', test_pred.mean())

Prediction min: 2.1708667136101726e-05
Prediction max: 0.9998454377684852
Prediction mean: 0.1751710290829895


## 5. Create Submission

In [5]:
submission = pd.DataFrame({
    'id': test_ids,
    'Will_Buy_EV': test_pred
})

output_path = '../submissions/submission_12.csv'
submission.to_csv(output_path, index=False)

print(submission.head())
print('Submission shape:', submission.shape)
print('Saved to:', output_path)

       id  Will_Buy_EV
0  668665     0.029555
1  668666     0.017700
2  668667     0.002924
3  668668     0.003479
4  668669     0.039701
Submission shape: (286571, 2)
Saved to: ../submissions/submission_12.csv


## 6. Verify Submission

In [6]:
assert submission.shape[0] == test.shape[0]
assert list(submission.columns) == ['id', 'Will_Buy_EV']
assert submission['Will_Buy_EV'].notna().all()
assert submission['Will_Buy_EV'].between(0, 1).all()
assert submission['id'].equals(test_ids)

print('All submission checks passed.')
print('Final file: ../submissions/submission_12.csv')

All submission checks passed.
Final file: ../submissions/submission_12.csv
